In [1]:
#first have to one hot encode the categorical columns
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split


In [2]:
df = pd.read_csv("/Users/sa19/Downloads/PS_20174392719_1491204439457_log.csv")

In [3]:
df_encoded = pd.get_dummies(df,columns=["type"],drop_first=True,dtype=int)

Encoding the type column that is categorical to integers to feed into the data. I also encoded through get dummies to drop the less relevant feauture. 

In [4]:
df_encoded.head(100)

,step,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.00,0.00,0,0,0,0,1,0
1,1,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.00,0.00,0,0,0,0,1,0
2,1,181.00,C1305486145,181.0,0.00,C553264065,0.00,0.00,1,0,0,0,0,1
3,1,181.00,C840083671,181.0,0.00,C38997010,21182.00,0.00,1,0,1,0,0,0
4,1,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.00,0.00,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1,710544.77,C835773569,0.0,0.00,C1359044626,738531.50,16518.36,0,0,0,0,0,1
96,1,581294.26,C843299092,0.0,0.00,C1590550415,5195482.15,19169204.93,0,0,0,0,0,1
97,1,11996.58,C605982374,0.0,0.00,C1225616405,40255.00,0.00,0,0,0,0,0,1
98,1,2875.10,C1412322831,15443.0,12567.90,M1651262695,0.00,0.00,0,0,0,0,1,0


In [5]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 14 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   amount          float64
 2   nameOrig        object 
 3   oldbalanceOrg   float64
 4   newbalanceOrig  float64
 5   nameDest        object 
 6   oldbalanceDest  float64
 7   newbalanceDest  float64
 8   isFraud         int64  
 9   isFlaggedFraud  int64  
 10  type_CASH_OUT   int64  
 11  type_DEBIT      int64  
 12  type_PAYMENT    int64  
 13  type_TRANSFER   int64  
dtypes: float64(5), int64(7), object(2)
memory usage: 679.6+ MB


In [6]:
df_encoded.drop(columns=["step","nameOrig","nameDest"], inplace=True)

I dropeed step because this column is not as significant to support my hypothesis. The nameOrig and nameDest are object types and these columns do not add significance to the model. However, how will models identify the account if there aren't any names? 

In [7]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   amount          float64
 1   oldbalanceOrg   float64
 2   newbalanceOrig  float64
 3   oldbalanceDest  float64
 4   newbalanceDest  float64
 5   isFraud         int64  
 6   isFlaggedFraud  int64  
 7   type_CASH_OUT   int64  
 8   type_DEBIT      int64  
 9   type_PAYMENT    int64  
 10  type_TRANSFER   int64  
dtypes: float64(5), int64(6)
memory usage: 534.0 MB


I will create a new feature called "BalanceDiff". This will be the difference between oldbalanceOrg and NewbalanceOrg. It will showcase the relationship between the balance difference and the amount. This is a strong indicator of fraud.

In [8]:
df_encoded['BalanceDiff'] = df["newbalanceOrig"] - df["oldbalanceOrg"]

In [9]:
df_encoded.drop(columns=["oldbalanceOrg","newbalanceOrig"], inplace=True)

In [10]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 10 columns):
 #   Column          Dtype  
---  ------          -----  
 0   amount          float64
 1   oldbalanceDest  float64
 2   newbalanceDest  float64
 3   isFraud         int64  
 4   isFlaggedFraud  int64  
 5   type_CASH_OUT   int64  
 6   type_DEBIT      int64  
 7   type_PAYMENT    int64  
 8   type_TRANSFER   int64  
 9   BalanceDiff     float64
dtypes: float64(4), int64(6)
memory usage: 485.4 MB


In [11]:
df_encoded.to_csv("/Users/sa19/class-projects/Fraud Detection Lab/Data.ipynb/Processed/encoded_data.csv", index=False)

With this dataset, I will keep the outliers because a bank holds a vast array of accounts with different balances. The outliers provide details that make up the dataset. I will choose Random Forest as a model because it is robust to outliers  

In [12]:
#Random Forest with random grid search for hyperparameter tuning. However, dataset has a small representayion of fraud samples and need a decent amount to be in the sample. " look in notes what this is called"